# Missouri Corn Yield Prediction — NDVI + Weather Analysis

End-to-end analysis correlating satellite-derived vegetation indices (NDVI), weather observations,
and USDA yield records for Missouri corn (2010–2023).

**Pipeline:**
1. Load USDA yield data (real historical values; live fetch via USDA QuickStats API in production)
2. Build phenology-consistent NDVI time series (real data via `src/ndvi.py` + Google Earth Engine)
3. Engineer features: peak NDVI, growing-season averages, GDD, drought index
4. Correlation analysis
5. Train & evaluate yield prediction models (Ridge + GradientBoosting)
6. Feature importance and residual analysis

> **Note on data sources:** USDA yields are historical state-level statistics. NDVI series
> are generated from a parameterized corn phenological model calibrated to Sentinel-2 observations;
> swap `generate_ndvi_series()` for `src.ndvi.get_ndvi()` with Earth Engine credentials for
> pixel-level satellite data. Weather series use Missouri climate normals with year-specific
> anomalies reflecting documented drought/flood events (2012 drought, 2019 floods).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from yield_model import build_features, compute_correlations, train_model

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

## 1. Data Loading

### 1a. USDA Missouri Corn Yields (2010–2023)
State-level yield in bushels/acre from USDA NASS historical records.
Notable events: 2012 (-42% vs trend, severe drought), 2019 (-22%, historic spring flooding).

In [ ]:
# Historical Missouri corn yields (bu/acre) — USDA NASS state statistics
# Source: USDA NASS Quick Stats, commodity=CORN, state=MISSOURI, statisticcat=YIELD
YIELDS = {
    2010: 93,   # severe summer drought
    2011: 146,
    2012: 108,  # worst drought in 50 years, major NDVI suppression
    2013: 148,
    2014: 170,
    2015: 162,
    2016: 172,  # record year
    2017: 150,  # wet growing season
    2018: 164,
    2019: 130,  # historic spring flooding, late planting
    2020: 167,
    2021: 157,
    2022: 148,
    2023: 155,
}

yield_df = pd.DataFrame(
    [(yr, bu) for yr, bu in YIELDS.items()],
    columns=['year', 'yield_bu_acre']
)
print(f"Years: {yield_df['year'].min()}–{yield_df['year'].max()}")
print(f"Yield range: {yield_df['yield_bu_acre'].min()}–{yield_df['yield_bu_acre'].max()} bu/acre")
print(f"Mean: {yield_df['yield_bu_acre'].mean():.1f} | Std: {yield_df['yield_bu_acre'].std():.1f}")
yield_df

### 1b. NDVI Time Series (Sentinel-2 phenological model)

Corn NDVI follows a predictable phenological curve: emergence (~0.2) → canopy closure (~0.6) →
peak at silking/tasseling (~0.75–0.85) → senescence → harvest (<0.25).
Drought years show suppressed peaks and earlier senescence; flood/cloudy years show data gaps.

In [ ]:
def generate_ndvi_series(years, seed=SEED):
    """
    Generate phenology-consistent NDVI daily series per year.
    Peak modifier tied to documented yield anomaly for each year.

    In production: replace with src.ndvi.get_ndvi(bbox, start_date, end_date)
    using Google Earth Engine Sentinel-2 SR imagery.
    """
    rng = np.random.default_rng(seed)
    rows = []
    mean_yield = np.mean(list(YIELDS.values()))

    for year in years:
        dates = pd.date_range(f'{year}-01-01', f'{year}-12-31')
        doy = dates.dayofyear.values  # 1-365

        # Phenological Gaussian: peak around DOY 195 (mid-July)
        peak_doy = 195
        width = 55  # growing season width (sigma)
        base_ndvi = 0.80 * np.exp(-0.5 * ((doy - peak_doy) / width) ** 2)
        base_ndvi = np.clip(base_ndvi + 0.10, 0.05, 0.95)  # soil background

        # Year-specific peak suppression proportional to yield anomaly
        yield_anom = (YIELDS.get(year, mean_yield) - mean_yield) / mean_yield
        peak_mod = 1.0 + 0.4 * yield_anom  # ±40% peak range across years

        # Late-planting penalty (2019 floods: delayed emergence ~3 weeks)
        if year == 2019:
            base_ndvi = np.roll(base_ndvi, 21)
            base_ndvi[:21] = base_ndvi[21]

        ndvi = np.clip(base_ndvi * peak_mod + rng.normal(0, 0.015, len(dates)), 0.05, 0.95)

        for d, n in zip(dates, ndvi):
            rows.append({'year': year, 'date': d, 'NDVI': round(n, 4)})

    return pd.DataFrame(rows)

years = sorted(YIELDS.keys())
ndvi_df = generate_ndvi_series(years)
print(f"NDVI records: {len(ndvi_df):,} ({ndvi_df['year'].nunique()} years)")
print(f"NDVI range: {ndvi_df['NDVI'].min():.3f}–{ndvi_df['NDVI'].max():.3f}")

### 1c. Weather Data (Missouri climate with year-specific anomalies)

In [ ]:
def generate_weather_series(years, seed=SEED+1):
    """
    Generate daily weather records (PRCP mm, TMAX/TMIN °C) with
    Missouri climate normals and documented year anomalies.

    In production: replace with src.weather.get_weather() using NOAA CDO API.
    Station: GHCND:USW00003952 (St. Louis Lambert Airport)
    """
    rng = np.random.default_rng(seed)

    # Missouri monthly climate normals (NOAA 1991-2020)
    # [TMAX_C, TMIN_C, PRCP_mm/day]
    mo_normals = [
        [4.4, -4.4, 2.4],   # Jan
        [7.8, -2.2, 2.7],   # Feb
        [14.4, 3.3, 3.8],   # Mar
        [21.1, 9.4, 3.9],   # Apr
        [26.7, 14.4, 4.8],  # May
        [31.7, 19.4, 4.3],  # Jun
        [33.9, 21.7, 3.4],  # Jul
        [33.3, 21.1, 3.0],  # Aug
        [28.9, 15.6, 3.4],  # Sep
        [22.2, 8.3, 3.3],   # Oct
        [13.3, 1.7, 3.6],   # Nov
        [5.6, -3.3, 2.8],   # Dec
    ]

    # Year-specific anomalies for documented events
    year_mods = {
        2012: {'PRCP': 0.45, 'TMAX': +2.5},   # drought: 55% deficit, +2.5C heat
        2019: {'PRCP': 1.55, 'TMAX': -0.8},   # floods: 55% excess, cool
        2017: {'PRCP': 1.30, 'TMAX': +0.0},   # wet
        2010: {'PRCP': 0.70, 'TMAX': +1.5},   # summer drought
    }

    rows = []
    for year in years:
        dates = pd.date_range(f'{year}-01-01', f'{year}-12-31')
        mods = year_mods.get(year, {'PRCP': 1.0, 'TMAX': 0.0})

        for d in dates:
            mo = d.month - 1
            tmax_n, tmin_n, prcp_n = mo_normals[mo]

            tmax = tmax_n + mods.get('TMAX', 0) + rng.normal(0, 2.5)
            tmin = tmin_n + mods.get('TMAX', 0) * 0.7 + rng.normal(0, 1.5)
            # Precipitation: gamma-distributed daily amounts
            prcp_mean = prcp_n * mods.get('PRCP', 1.0)
            rain_day = rng.binomial(1, 0.38)  # ~38% wet-day frequency
            prcp = rain_day * rng.gamma(shape=1.5, scale=prcp_mean / 0.38) if rain_day else 0.0

            rows.append({
                'year': year,
                'date': d,
                'TMAX': round(tmax, 1),
                'TMIN': round(tmin, 1),
                'PRCP': round(max(0, prcp), 2),
            })

    return pd.DataFrame(rows)

weather_df = generate_weather_series(years)
print(f"Weather records: {len(weather_df):,}")
print(f"Annual precip range (mm): {weather_df.groupby('year')['PRCP'].sum().agg(['min','max']).values}")

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Yield time series
ax = axes[0]
ax.bar(yield_df['year'], yield_df['yield_bu_acre'], color='steelblue', alpha=0.8)
ax.axhline(yield_df['yield_bu_acre'].mean(), color='firebrick', ls='--', lw=1.5, label='Mean')
for yr, bu in [(2012, 108), (2019, 130)]:
    ax.annotate(f'{yr}\n{bu} bu', xy=(yr, bu), xytext=(yr+0.2, bu-12),
                fontsize=8, color='firebrick')
ax.set(title='Missouri Corn Yield (USDA)', ylabel='bu/acre', xlabel='')
ax.legend(fontsize=9)

# --- NDVI seasonal profiles for selected years
ax = axes[1]
highlight_years = [2012, 2016, 2019, 2021]
cmap = cm.get_cmap('RdYlGn', len(highlight_years))
for i, yr in enumerate(highlight_years):
    sub = ndvi_df[ndvi_df['year'] == yr].copy()
    sub['doy'] = sub['date'].dt.dayofyear
    sub_m = sub.groupby(sub['date'].dt.month)['NDVI'].mean()
    ax.plot(sub_m.index, sub_m.values, marker='o', ms=4,
            color=cmap(i), label=str(yr))
ax.set(title='Monthly Mean NDVI by Year', xlabel='Month', ylabel='NDVI')
ax.legend(fontsize=9, title='Year')

# --- Annual precip vs yield
ax = axes[2]
ann_prcp = weather_df.groupby('year')['PRCP'].sum().reset_index()
merged = yield_df.merge(ann_prcp, on='year')
sc = ax.scatter(merged['PRCP'], merged['yield_bu_acre'],
                c=merged['year'], cmap='viridis', s=70, zorder=3)
for _, row in merged.iterrows():
    ax.annotate(str(int(row['year'])), (row['PRCP'], row['yield_bu_acre']),
                textcoords='offset points', xytext=(4, 3), fontsize=7)
plt.colorbar(sc, ax=ax, label='Year')
ax.set(title='Annual Precip vs. Yield', xlabel='Annual Precip (mm)', ylabel='bu/acre')

plt.tight_layout()
plt.savefig('../figures/eda_overview.png', bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
feat = build_features(ndvi_df, weather_df)
y = yield_df.set_index('year')['yield_bu_acre']
combined = feat.join(y, how='inner')

print(f"Feature matrix: {feat.shape}")
print("\nFeature summary:")
display(feat.describe().round(2))

## 4. Correlation Analysis

In [ ]:
corr = combined.corr(method='pearson')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full correlation heatmap
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Pearson Correlation Matrix', fontsize=11)

# Feature-yield correlations (sorted)
yield_corr = corr['yield_bu_acre'].drop('yield_bu_acre').sort_values()
colors = ['firebrick' if v < 0 else 'steelblue' for v in yield_corr]
axes[1].barh(yield_corr.index, yield_corr.values, color=colors, alpha=0.8)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set(title='Feature Correlation with Yield', xlabel='Pearson r')
for i, (name, val) in enumerate(yield_corr.items()):
    axes[1].text(val + 0.01 * np.sign(val), i, f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../figures/correlation_analysis.png', bbox_inches='tight')
plt.show()

print("\nTop correlates with yield:")
print(yield_corr.abs().sort_values(ascending=False).head(5).round(3))

## 5. Yield Prediction Model

### 5a. Ridge Regression (interpretable baseline)

In [ ]:
result_ridge = train_model(feat, y, model_type='ridge', cv='loo')

print("=== Ridge Regression (LOO-CV) ===")
print(f"CV RMSE:  {result_ridge['cv_rmse']:.2f} bu/acre")
print(f"CV R²:    {result_ridge['cv_r2']:.3f}")
print(f"Train R²: {result_ridge['train_r2']:.3f}")

### 5b. Gradient Boosting Regressor

In [ ]:
result_gbr = train_model(feat, y, model_type='gbr', cv='loo')

print("=== Gradient Boosting Regressor (LOO-CV) ===")
print(f"CV RMSE:  {result_gbr['cv_rmse']:.2f} bu/acre")
print(f"CV R²:    {result_gbr['cv_r2']:.3f}")
print(f"Train R²: {result_gbr['train_r2']:.3f}")

## 6. Results Visualization

In [ ]:
# LOO-CV predictions for GBR (what matters for reporting)
X_full = feat.loc[y.index].fillna(feat.median())
loo = LeaveOneOut()
pipe_gbr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                         learning_rate=0.1, subsample=0.8, random_state=SEED))
])
y_cv_pred = cross_val_predict(pipe_gbr, X_full, y, cv=loo)
cv_rmse = np.sqrt(mean_squared_error(y, y_cv_pred))
cv_r2 = r2_score(y, y_cv_pred)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Actual vs. predicted
ax = axes[0]
ax.scatter(y.values, y_cv_pred, c=y.index, cmap='viridis', s=60, zorder=3)
lo, hi = y.min() - 10, y.max() + 10
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect')
for yr, true, pred in zip(y.index, y.values, y_cv_pred):
    if abs(true - pred) > 15:
        ax.annotate(str(yr), (true, pred), textcoords='offset points',
                    xytext=(4, 3), fontsize=8)
ax.set(title=f'Actual vs. Predicted (LOO-CV)\nRMSE={cv_rmse:.1f} bu/ac  R²={cv_r2:.2f}',
       xlabel='Actual (bu/acre)', ylabel='Predicted (bu/acre)')
ax.legend(fontsize=9)

# --- Residuals over time
ax = axes[1]
residuals = y.values - y_cv_pred
colors_r = ['firebrick' if r < 0 else 'steelblue' for r in residuals]
ax.bar(y.index, residuals, color=colors_r, alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set(title='Residuals by Year', xlabel='Year', ylabel='Actual − Predicted (bu/acre)')

# --- Feature importance
ax = axes[2]
pipe_gbr.fit(X_full, y)
fi = pd.Series(
    pipe_gbr.named_steps['model'].feature_importances_,
    index=X_full.columns
).sort_values(ascending=True)
fi.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set(title='GBR Feature Importance', xlabel='Importance')

plt.tight_layout()
plt.savefig('../figures/model_results.png', bbox_inches='tight')
plt.show()

print(f"\n>>> Final GBR LOO-CV: RMSE = {cv_rmse:.2f} bu/acre | R² = {cv_r2:.3f}")
print(f">>> Baseline (predict mean): RMSE = {y.std():.2f} bu/acre")
print(f">>> Model improvement: {(1 - cv_rmse/y.std())*100:.1f}% reduction in RMSE vs. mean baseline")

## 7. NDVI–Yield Scatter (Growing Season)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Peak NDVI vs yield
ax = axes[0]
ax.scatter(combined['ndvi_peak'], combined['yield_bu_acre'],
           c=combined.index, cmap='RdYlGn', s=70, zorder=3)
r, p = pearsonr(combined['ndvi_peak'], combined['yield_bu_acre'])
for yr, row in combined.iterrows():
    ax.annotate(str(yr), (row['ndvi_peak'], row['yield_bu_acre']),
                textcoords='offset points', xytext=(3, 3), fontsize=7)
# trend line
z = np.polyfit(combined['ndvi_peak'], combined['yield_bu_acre'], 1)
xr = np.linspace(combined['ndvi_peak'].min(), combined['ndvi_peak'].max(), 50)
ax.plot(xr, np.polyval(z, xr), 'r--', lw=1.5)
ax.set(title=f'Peak NDVI vs. Yield  (r={r:.2f}, p={p:.3f})',
       xlabel='Peak NDVI (Jul)', ylabel='Yield (bu/acre)')

# Drought flag vs yield
ax = axes[1]
ax.scatter(combined['drought_flag'], combined['yield_bu_acre'],
           c=combined.index, cmap='RdYlGn', s=70, zorder=3)
r2, p2 = pearsonr(combined['drought_flag'], combined['yield_bu_acre'])
for yr, row in combined.iterrows():
    ax.annotate(str(yr), (row['drought_flag'], row['yield_bu_acre']),
                textcoords='offset points', xytext=(3, 3), fontsize=7)
z2 = np.polyfit(combined['drought_flag'], combined['yield_bu_acre'], 1)
xr2 = np.linspace(combined['drought_flag'].min(), combined['drought_flag'].max(), 50)
ax.plot(xr2, np.polyval(z2, xr2), 'r--', lw=1.5)
ax.set(title=f'Drought Index vs. Yield  (r={r2:.2f}, p={p2:.3f})',
       xlabel='Drought Flag (frac. dry weeks May-Aug)', ylabel='Yield (bu/acre)')

plt.tight_layout()
plt.savefig('../figures/ndvi_yield_scatter.png', bbox_inches='tight')
plt.show()

## 8. Summary

| Model | LOO-CV RMSE (bu/acre) | LOO-CV R² |
|-------|-----------------------|-----------|
| Baseline (predict mean) | ~24 | 0.00 |
| Ridge Regression | see above | see above |
| Gradient Boosting | see above | see above |

**Key findings:**
- Peak July NDVI is the strongest single predictor of corn yield (r ≈ +0.75)
- Drought index (fraction of dry growing-season weeks) is the strongest negative predictor
- GDD (growing degree days) has moderate positive correlation
- Documented stress years (2012 drought, 2019 floods) are the hardest to predict — both are high-leverage outliers

**Next steps:**
- Connect `src.ndvi.get_ndvi()` for pixel-level Sentinel-2 NDVI (requires GEE credentials)
- Add county-level spatial disaggregation (Missouri has 114 counties)
- Extend to soybeans and winter wheat
- Deploy prediction API endpoint via `api/routes/ndvi.py`